# Lesson 2: Data Engineering — Homework

В реальному світі документи бувають зламані, з неправильним кодуванням, порожні, з підмінеими розширеннями. Ваша задача — навчитись їх обробляти.

**6 завдань.** Кожне має `# TODO` — дописуйте реалізацію.

```bash
pip install -r requirements.txt
```

In [ ]:
import os
from pathlib import Path

# Генеруємо тестові документи
if not Path("samples/enterprise_challenges").exists():
    os.system("python src/generate_bad_samples.py")

print("Файли для роботи:")
for f in sorted(Path("samples/enterprise_challenges").iterdir()):
    size = f.stat().st_size
    print(f"  {f.name:<40} {size:>10,} bytes")

---
## Завдання 1: Визначення кодування файлів

Legacy-системи часто віддають файли в Windows-1251, Latin-1, UTF-8 з BOM — без вказання charset.
Якщо відкрити такий файл з неправильним кодуванням, отримаємо кракозябри (mojibake).

**Файли для роботи:**
- `utf8_with_bom.html` — UTF-8 з BOM маркером (зайві байти `\xef\xbb\xbf` на початку)
- `windows1251_no_charset.html` — Windows-1251 без мета-тегу charset (українські legacy-системи)
- `latin1_mixed.html` — Latin-1 з німецькими/французькими символами

**Що потрібно зробити:**
1. Прочитати файл як bytes
2. Визначити кодування через `charset_normalizer`
3. Декодувати текст правильно
4. Якщо є BOM — прибрати його

In [ ]:
from charset_normalizer import from_bytes


def detect_and_read(file_path: str) -> dict:
    """
    Читає файл, визначає кодування, повертає декодований текст.

    Повертає:
    {
        "file": ім'я файлу,
        "encoding": визначене кодування,
        "had_bom": True/False,
        "text": декодований текст,
        "char_count": кількість символів
    }
    """
    file_path = Path(file_path)
    raw = file_path.read_bytes()

    # TODO: перевірити чи файл починається з BOM (b"\xef\xbb\xbf")
    # Якщо так — прибрати BOM з raw bytes
    had_bom = False

    # TODO: використати from_bytes(raw).best() для визначення кодування
    # result.encoding — назва кодування, str(result) — декодований текст
    # Якщо best() повернув None — декодувати як utf-8 з errors="replace"
    encoding = None
    text = ""

    return {
        "file": file_path.name,
        "encoding": encoding,
        "had_bom": had_bom,
        "text": text,
        "char_count": len(text),
    }

In [ ]:
# Тест
encoding_files = [
    "samples/enterprise_challenges/utf8_with_bom.html",
    "samples/enterprise_challenges/windows1251_no_charset.html",
    "samples/enterprise_challenges/latin1_mixed.html",
]

for f in encoding_files:
    result = detect_and_read(f)
    print(f"{result['file']}:")
    print(f"  Кодування: {result['encoding']}")
    print(f"  BOM: {result['had_bom']}")
    print(f"  Перші 150 символів: {result['text'][:150]}")
    print()

---
## Завдання 2: Визначення реального типу файлу (magic bytes)

Розширення файлу може брехати. Хтось зберіг HTML як `.pdf`, або binary dump перейменував в `.pdf`.
Єдиний надійний спосіб — перевірити **magic bytes** (перші байти файлу, які визначають формат).

**Файли для роботи:**
- `actually_html.pdf` — HTML контент зі розширенням .pdf
- `actually_pdf.html` — PDF контент зі розширенням .html
- `binary_garbage.pdf` — рандомні байти з розширенням .pdf
- `empty_file.pdf` — порожній файл (0 байт)

**Що потрібно зробити:**
1. Визначити тип файлу по розширенню (declared type)
2. Визначити реальний тип через бібліотеку `filetype` (magic bytes)
3. Для HTML файлів (без magic bytes) — перевірити чи контент починається з `<html` або `<!doctype`
4. Повернути чи є mismatch

In [ ]:
import filetype as filetype_lib


def detect_file_type(file_path: str) -> dict:
    """
    Визначає реальний тип файлу і порівнює з розширенням.

    Повертає:
    {
        "file": ім'я файлу,
        "declared_type": тип по розширенню,
        "detected_type": реальний тип (magic bytes),
        "is_mismatch": True якщо типи не збігаються,
        "issue": опис проблеми або None
    }
    """
    file_path = Path(file_path)
    declared_type = file_path.suffix.lstrip(".").lower()

    # TODO: перевірити чи файл порожній — якщо так, повернути issue "empty file"

    # TODO: використати filetype_lib.guess(str(file_path)) для визначення типу
    # guess() повертає об'єкт з .extension або None якщо не вдалось визначити
    detected_type = None

    # TODO: якщо filetype не визначив (None) — перевірити вручну чи це HTML
    # Прочитати перші 512 байт і перевірити чи є b"<html" або b"<!doctype"

    # TODO: визначити чи є mismatch між declared_type і detected_type
    is_mismatch = False
    issue = None

    return {
        "file": file_path.name,
        "declared_type": declared_type,
        "detected_type": detected_type,
        "is_mismatch": is_mismatch,
        "issue": issue,
    }

In [ ]:
# Тест
test_files = [
    "samples/enterprise_challenges/actually_html.pdf",
    "samples/enterprise_challenges/actually_pdf.html",
    "samples/enterprise_challenges/binary_garbage.pdf",
    "samples/enterprise_challenges/empty_file.pdf",
    "samples/enterprise_challenges/normal_report.xlsx",
]

for f in test_files:
    if Path(f).exists():
        result = detect_file_type(f)
        marker = "MISMATCH" if result["is_mismatch"] else "OK"
        print(f"  [{marker}] {result['file']}: declared=.{result['declared_type']}, real={result['detected_type']}")
        if result["issue"]:
            print(f"          -> {result['issue']}")

---
## Завдання 3: Витягування тексту з брудного HTML

Enterprise HTML — це жах: 50 рівнів вкладеності від Word-експорту, navigation bars, sidebars, scripts, cookie banners. Корисного тексту — 5%.

**Файли для роботи:**
- `malformed_deeply_nested.html` — 50 рівнів `<div>`, незакриті теги, зламані entities
- `boilerplate_heavy.html` — 30 nav items, 20 sidebar widgets, 3 абзаци корисного тексту

**Що потрібно зробити:**
1. Прочитати HTML через BeautifulSoup
2. Видалити `<script>`, `<style>`, `<nav>`, `<header>`, `<footer>` теги
3. Витягти чистий текст
4. Порахувати "корисність" — відсоток тексту від розміру HTML

In [ ]:
from bs4 import BeautifulSoup


def extract_clean_text(file_path: str) -> dict:
    """
    Витягує чистий текст з HTML, видаляючи шум.

    Повертає:
    {
        "file": ім'я файлу,
        "raw_size": розмір HTML в символах,
        "text": чистий текст,
        "text_size": розмір тексту в символах,
        "useful_ratio": відсоток корисного тексту (text_size / raw_size)
    }
    """
    file_path = Path(file_path)
    raw_html = file_path.read_text(errors="replace")

    # TODO: створити BeautifulSoup з парсером "html.parser"
    soup = ...

    # TODO: видалити всі теги які є шумом: script, style, nav, header, footer, aside
    # Підказка: soup.find_all("script") повертає список, для кожного — tag.decompose()
    noise_tags = ["script", "style", "nav", "header", "footer", "aside"]

    # TODO: витягти текст через soup.get_text(separator="\n", strip=True)
    text = ""

    raw_size = len(raw_html)
    text_size = len(text)
    useful_ratio = text_size / raw_size if raw_size > 0 else 0

    return {
        "file": file_path.name,
        "raw_size": raw_size,
        "text": text,
        "text_size": text_size,
        "useful_ratio": useful_ratio,
    }

In [ ]:
# Тест
html_files = [
    "samples/enterprise_challenges/malformed_deeply_nested.html",
    "samples/enterprise_challenges/boilerplate_heavy.html",
    "samples/enterprise_challenges/multilingual.html",
]

for f in html_files:
    result = extract_clean_text(f)
    print(f"{result['file']}:")
    print(f"  HTML: {result['raw_size']:,} символів -> Текст: {result['text_size']:,} символів")
    print(f"  Корисність: {result['useful_ratio']:.1%}")
    print(f"  Текст: {result['text'][:200]}...")
    print()

---
## Завдання 4: Обробка зламаних файлів — safe parser

Файли ламаються: обрізані PDF, corrupted DOCX (зламаний ZIP), binary garbage з розширенням .pdf, порожні файли, захищені паролем PDF.

Парсер не повинен падати — він має повертати результат або зрозумілу помилку.

**Файли для роботи:** вся папка `enterprise_challenges/` — там є і робочі, і зламані файли.

**Що потрібно зробити:**
1. Перевірити файл (порожній? правильний тип?)
2. Спробувати спарсити через `unstructured.partition.auto.partition`
3. Якщо помилка — зловити exception і повернути опис проблеми
4. Класифікувати помилку: `empty`, `corrupted`, `type_mismatch`, `parse_error`

In [ ]:
from unstructured.partition.auto import partition


def safe_parse(file_path: str) -> dict:
    """
    Безпечно парсить документ — ніколи не падає, завжди повертає результат.

    Повертає при успіху:
    {
        "file": ім'я, "status": "ok",
        "text": витягнутий текст, "char_count": кількість символів
    }

    Повертає при помилці:
    {
        "file": ім'я, "status": "error",
        "error_type": тип помилки (empty/corrupted/type_mismatch/parse_error),
        "error_message": опис
    }
    """
    file_path = Path(file_path)

    # TODO: перевірити чи файл порожній (0 байт)
    # Якщо так — повернути status="error", error_type="empty"

    # TODO: перевірити тип файлу через detect_file_type() з завдання 2
    # Якщо mismatch — повернути status="error", error_type="type_mismatch"

    # TODO: спробувати спарсити файл через partition(filename=str(file_path))
    # Обгорнути в try/except Exception
    # При успіху — з'єднати елементи в текст і повернути status="ok"
    # При помилці — повернути status="error", error_type="parse_error" або "corrupted"

    return {"file": file_path.name, "status": "error", "error_type": "not_implemented", "error_message": "TODO"}

In [ ]:
# Тест: прогнати safe_parse по всій папці enterprise_challenges
results = []
challenges_dir = Path("samples/enterprise_challenges")

for f in sorted(challenges_dir.iterdir()):
    if f.is_file():
        result = safe_parse(str(f))
        results.append(result)

ok = [r for r in results if r["status"] == "ok"]
errors = [r for r in results if r["status"] == "error"]

print(f"=== Результати: {len(ok)} OK, {len(errors)} ERROR ===\n")

print("OK:")
for r in ok:
    print(f"  {r['file']}: {r['char_count']} символів")

print(f"\nERROR:")
for r in errors:
    print(f"  {r['file']}: [{r['error_type']}] {r['error_message']}")

---
## Завдання 5: Dead Letter Queue

В production системах файли які не вдалось обробити не ігноруються — вони потрапляють у **Dead Letter Queue (DLQ)**. Це окрема папка куди зберігаються "мертві" файли + error report, щоб потім розібратись що пішло не так.

**Що потрібно зробити:**
1. Реалізувати клас `DeadLetterQueue` — копіює зламаний файл + зберігає error report (JSON)
2. Реалізувати `process_all` — обробляє всі файли, робочі зберігає як результат, зламані кидає в DLQ
3. Вивести фінальну статистику

In [ ]:
import shutil
import json


class DeadLetterQueue:
    """
    Dead Letter Queue — зберігає файли які не вдалось обробити.

    Для кожного файлу:
    - копіює його в dlq_dir
    - зберігає {filename}_error.json з описом помилки
    """

    def __init__(self, dlq_dir: str = "data/dlq"):
        self.dlq_dir = Path(dlq_dir)
        self.dlq_dir.mkdir(parents=True, exist_ok=True)
        self.errors = []

    def send(self, file_path: str, error_type: str, error_message: str):
        """
        Відправити файл в DLQ.
        """
        file_path = Path(file_path)

        # TODO: скопіювати файл в self.dlq_dir через shutil.copy2()
        # (перевірте чи файл існує перед копіюванням)

        # TODO: створити error_report — словник з полями:
        #   file, error_type, error_message
        error_report = {}

        # TODO: зберегти error_report як JSON у self.dlq_dir / f"{file_path.stem}_error.json"

        # TODO: додати error_report в self.errors

    def list_errors(self) -> list[dict]:
        return self.errors

    @property
    def count(self) -> int:
        return len(self.errors)


def process_all(input_dir: str) -> dict:
    """
    Обробляє всі файли з папки:
    - Робочі → список результатів
    - Зламані → Dead Letter Queue
    """
    input_dir = Path(input_dir)
    dlq = DeadLetterQueue()
    processed = []

    for f in sorted(input_dir.iterdir()):
        if not f.is_file():
            continue

        # TODO: викликати safe_parse() для кожного файлу
        # Якщо status == "ok" — додати в processed
        # Якщо status == "error" — відправити в dlq через dlq.send()
        pass

    return {
        "processed": processed,
        "dlq": dlq,
        "stats": {
            "total": len(processed) + dlq.count,
            "success": len(processed),
            "dead_letters": dlq.count,
        },
    }

In [ ]:
# Тест
result = process_all("samples/enterprise_challenges")

print(f"=== Статистика ===")
print(f"  Всього:        {result['stats']['total']}")
print(f"  Успішно:       {result['stats']['success']}")
print(f"  Dead letters:  {result['stats']['dead_letters']}")

print(f"\n=== Dead Letter Queue ===")
for err in result["dlq"].list_errors():
    print(f"  {err['file']}: [{err['error_type']}] {err['error_message']}")

print(f"\n=== Успішно оброблені ===")
for r in result["processed"]:
    print(f"  {r['file']}: {r['char_count']} символів")

print(f"\nDLQ файли збережені в: {result['dlq'].dlq_dir}/")
print(f"Перевірте: ls {result['dlq'].dlq_dir}/")

---
## Завдання 6: Chunking великого документа

Для RAG та embeddings текст треба розбити на чанки. Але з великим файлом (500K+ символів) є нюанси:
- Як розмір чанка впливає на кількість чанків?
- Що відбувається з overlap?
- Скільки часу займає chunking?

**Що потрібно зробити:**
1. Згенерувати великий текстовий файл (~500K символів)
2. Реалізувати chunking через `RecursiveCharacterTextSplitter`
3. Порівняти різні `chunk_size` (256, 512, 1024, 2048) — кількість чанків, час, overlap
4. Порівняти різні `chunk_overlap` (0, 50, 200) — як overlap впливає на кількість чанків

In [ ]:
# Генеруємо великий файл (~500K символів) — імітація enterprise audit log
huge_file = Path("samples/enterprise_challenges/huge_audit_log.txt")

if not huge_file.exists():
    paragraphs = [
        "The quarterly financial review revealed significant growth across all business units. "
        "Revenue increased by 15% year-over-year, driven by expansion into Southeast Asian markets. "
        "Operating margins improved from 22% to 27%, reflecting cost optimization initiatives.",

        "Customer acquisition costs decreased by 12% while retention rates improved to 94%. "
        "The enterprise segment showed particularly strong performance with 45 new contracts signed. "
        "Average deal size increased from $120K to $185K, indicating successful upmarket movement.",

        "Engineering velocity metrics showed improvement: deployment frequency increased to 47 per week, "
        "change failure rate dropped to 2.1%, and mean time to recovery decreased to 23 minutes. "
        "Technical debt ratio was reduced from 18% to 11% through focused refactoring sprints.",

        "Compliance audit for Q4 completed successfully with zero critical findings. "
        "All PII data handling procedures meet GDPR and CCPA requirements. "
        "Encryption at rest upgraded to AES-256 across all production databases.",

        "Infrastructure costs optimized through reserved instance purchasing and auto-scaling improvements. "
        "Monthly cloud spend reduced from $340K to $285K while supporting 60% more traffic. "
        "CDN cache hit ratio improved to 97.3%, reducing origin server load significantly.",
    ]

    lines = []
    for i in range(5000):
        p = paragraphs[i % len(paragraphs)]
        lines.append(f"[Entry {i+1:05d}] {p}")

    huge_file.write_text("\n\n".join(lines))

size_mb = huge_file.stat().st_size / 1024 / 1024
text = huge_file.read_text()
print(f"Файл: {huge_file.name}")
print(f"Розмір: {size_mb:.2f} MB, {len(text):,} символів")

In [ ]:
import time
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_text(text: str, chunk_size: int = 512, chunk_overlap: int = 50) -> list[str]:
    """
    Розбиває текст на чанки.
    Повертає список рядків (чанків).
    """
    # TODO: створити RecursiveCharacterTextSplitter з chunk_size та chunk_overlap
    splitter = ...

    # TODO: розбити текст через splitter.split_text(text) і повернути результат
    chunks = ...

    return chunks

In [ ]:
# Експеримент 1: різний chunk_size (overlap=50)
print(f"Текст: {len(text):,} символів\n")
print(f"{'chunk_size':>12} | {'чанків':>8} | {'сер. довж.':>10} | {'час (мс)':>10}")
print("-" * 52)

for size in [256, 512, 1024, 2048]:
    t0 = time.time()
    chunks = chunk_text(text, chunk_size=size, chunk_overlap=50)
    elapsed = (time.time() - t0) * 1000
    avg = sum(len(c) for c in chunks) / len(chunks) if chunks else 0
    print(f"{size:>12} | {len(chunks):>8} | {avg:>10.0f} | {elapsed:>10.1f}")

In [ ]:
# Експеримент 2: різний chunk_overlap (chunk_size=512)
print(f"chunk_size=512, текст: {len(text):,} символів\n")
print(f"{'overlap':>12} | {'чанків':>8} | {'додатково':>12} | {'час (мс)':>10}")
print("-" * 52)

baseline = None
for overlap in [0, 50, 100, 200]:
    t0 = time.time()
    chunks = chunk_text(text, chunk_size=512, chunk_overlap=overlap)
    elapsed = (time.time() - t0) * 1000
    if baseline is None:
        baseline = len(chunks)
    extra = len(chunks) - baseline
    print(f"{overlap:>12} | {len(chunks):>8} | +{extra:>10} | {elapsed:>10.1f}")

print(f"\n(overlap збільшує кількість чанків, бо текст дублюється на стиках)")

In [ ]:
# Подивимось на стик між чанками — як працює overlap
chunks = chunk_text(text, chunk_size=512, chunk_overlap=100)
print(f"Всього чанків: {len(chunks)}\n")

print("=== Чанк 0 (останні 150 символів) ===")
print(f"...{chunks[0][-150:]}")
print(f"\n=== Чанк 1 (перші 150 символів) ===")
print(f"{chunks[1][:150]}...")
print(f"\n^ Бачите overlap? Кінець чанка 0 повторюється на початку чанка 1.")
print("  Це потрібно щоб контекст не губився на стиках.")

---
## Готово!

Ви навчились:
1. **Визначати кодування** — charset_normalizer, BOM, legacy encodings
2. **Перевіряти тип файлу** — magic bytes vs розширення
3. **Чистити HTML** — видаляти шум, витягувати корисний текст
4. **Безпечно парсити** — обробляти corrupted/empty/broken файли без crash
5. **Dead Letter Queue** — production-pattern для зламаних файлів
6. **Chunking** — як chunk_size і overlap впливають на результат для великих документів